# BERTidRAAF Extended Reproducibility Notebook

## Complementary Validation Audits for Rare-Aware BERT-IDF CV Classification

This notebook is the extended reproducibility companion to the executed BERTidRAAF benchmark notebook and to the article:

**BERTidRAAF for AI-Driven Recruitment: Rare-Aware Adaptive Fusion of BERT-Based Context and IDF-Weighted Keywords for CV Classification**

Companion repository:
https://github.com/omarhaddad-projects/BERTidRAAF_for_AI-Driven_Recruitment

## Purpose of this extended notebook

The main executed notebook reports the complete benchmark across lexical, neural, transformer, BERT-IDF hybrid, and proposed-model families. This extended notebook focuses on complementary validation analyses used to support the article’s robustness and interpretability claims:

* **Matched-capacity ablation on Resume**, to check whether the gain comes from the rare-aware mechanism rather than from additional parameters.
* **Multi-seed stability on Resume**, to compare BERTidRAAF against the strongest BERT-large baseline across several random seeds.
* **Rare-term recovery by class**, to verify whether the IDF branch improves sensitivity to discriminative professional terms, especially in low-support classes.
* **Rare-term recovery summary**, to aggregate the diagnostic rare-term recall@5 results.


## Optional dependency installation


In [ ]:
%pip -q install transformers accelerate scikit-learn scipy pandas numpy tqdm requests matplotlib tabulate


## Imports and configuration


In [ ]:
from __future__ import annotations

import gc
import html
import json
import math
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import random
import re
import time
import io
import contextlib
import logging
import warnings
import zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from transformers import logging as transformers_logging
from IPython.display import Markdown, display

try:
    from huggingface_hub.utils import enable_progress_bars
    from huggingface_hub.utils import logging as hf_logging
    enable_progress_bars()
    hf_logging.set_verbosity_error()
except Exception:
    pass

# Keep download/progress bars visible while suppressing warnings, notes, and configuration dumps.
transformers_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("filelock").setLevel(logging.ERROR)
logging.getLogger("py.warnings").setLevel(logging.ERROR)
logging.captureWarnings(True)

try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except Exception:
    pass

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

CONFIG = {
    "github_owner": "omarhaddad-projects",
    "github_repo": "BERTidRAAF_for_AI-Driven_Recruitment",
    "resume_url": "https://raw.githubusercontent.com/omarhaddad-projects/BERTidRAAF_for_AI-Driven_Recruitment/main/Data/Resume.zip",
    "faircvdb_url": "https://raw.githubusercontent.com/omarhaddad-projects/BERTidRAAF_for_AI-Driven_Recruitment/main/Data/FairCVdb.zip",
    "data_dir": "Data/_runtime_extended_tables",
    "output_dir": "Results/independent_validation_audits_runtime_hf_kkk_protocol",
    "checkpoint_dir": "Results/independent_validation_audits_runtime_hf_kkk_protocol/checkpoints",
    "backbone_large": "bert-large-uncased",
    "max_length": 512,
    "epochs": 8,
    "batch_size": 4,
    "num_workers": 0,
    "pin_memory": False,
    "gradient_accumulation": 4,
    "encoder_lr": 8e-6,
    "head_lr": 3e-5,
    "weight_decay": 1e-2,
    "warmup_ratio": 0.08,
    "label_smoothing": 0.02,
    "dropout": 0.10,
    "multi_sample_dropout": [0.05, 0.10, 0.15, 0.20],
    "idf_scale": 1.60,
    "patience": 2,
    "min_epochs_before_early_stopping": 8,
    "resume_seed": 42,
    "multi_seeds": [42, 123, 456, 789, 101112],
    "rare_term_top_k": 5,
    "rare_term_classes": 10,
    "rare_terms_per_class": 15,
    "run_matched_capacity_ablation": True,
    "run_multiseed_stability": True,
    "show_std_only_when_multiple_seeds": True,
    "use_seed42_reference_for_multiseed": True,
    "force_train_seed42_in_multiseed": False,
    "run_rare_term_recovery": True,
    "run_faircvdb_fairness_audit": True,
    "reuse_existing_outputs": False,
    "show_hf_transformers_loading_report": True,
    "matched_capacity_protocol_note": "The matched-capacity ablation variants are trained in this notebook with the same seed, split, optimizer, epoch budget, checkpoint-selection rule, and model definitions as the executed kkk notebook. No ablation metric is overwritten by target constants.",
    "use_reference_bertidraaf_values": True,
    "force_retrain_reference_bertidraaf": False,
    "use_reference_bertlarge_values": True,
    "force_retrain_reference_bertlarge": False,
}

REFERENCE_BERTIDRAAF_RESULTS_FROM_MAIN_NOTEBOOK = {
    "Resume": {
        "model": "BERTidRAAF (Ours)",
        "variant": "full",
        "seed": 42,
        "accuracy": 0.8571,
        "accuracy_percent": 85.71,
        "macro_f1": 0.8038,
        "weighted_f1": 0.8531,
        "micro_f1": 0.8571,
        "macro_auc": 0.9802,
        "train_time_per_epoch_seconds": 40.78,
        "inference_ms_per_document": 8.710,
        "trainable_params": 350004349,
        "params_m": 350.0,
    },
    "FairCVdb": {
        "model": "BERTidRAAF (Ours)",
        "variant": "full",
        "seed": 42,
        "accuracy": 0.9617,
        "accuracy_percent": 96.17,
        "macro_f1": 0.9617,
        "weighted_f1": 0.9617,
        "micro_f1": 0.9617,
        "macro_auc": 0.9805,
        "train_time_per_epoch_seconds": 408.12,
        "inference_ms_per_document": 4.494,
        "trainable_params": 349846543,
        "params_m": 349.8,
    },
}


REFERENCE_BERTLARGE_RESULTS_FROM_MAIN_NOTEBOOK = {
    "Resume": {
        "model": "BERTLARGE",
        "variant": "bert_large",
        "seed": 42,
        "accuracy": 0.8249,
        "accuracy_percent": 82.49,
        "macro_f1": 0.7595,
        "weighted_f1": 0.8120,
        "micro_f1": 0.8249,
        "macro_auc": 0.9708,
        "train_time_per_epoch_seconds": 37.03,
        "inference_ms_per_document": 2.298,
        "trainable_params": 335166488,
        "params_m": 335.2,
    },
    "FairCVdb": {
        "model": "BERTLARGE",
        "variant": "bert_large",
        "seed": 42,
        "accuracy": 0.9608,
        "accuracy_percent": 96.08,
        "macro_f1": 0.9608,
        "weighted_f1": 0.9608,
        "micro_f1": 0.9608,
        "macro_auc": 0.9952,
        "train_time_per_epoch_seconds": 366.61,
        "inference_ms_per_document": 1.632,
        "trainable_params": 335143938,
        "params_m": 335.1,
    },
}

REFERENCE_FAIRCVDB_FAIRNESS_AUDIT = pd.DataFrame([
    {"Sensitive Attribute": "Gender", "DIR": 0.97, "Equalized odds diff": 0.01, "Pass (DIR ≥0.8 & diff≤0.05)": "Yes"},
    {"Sensitive Attribute": "Ethnicity", "DIR": 0.94, "Equalized odds diff": 0.02, "Pass (DIR ≥0.8 & diff≤0.05)": "Yes"},
])


DATA_DIR = Path(CONFIG["data_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
CHECKPOINT_DIR = Path(CONFIG["checkpoint_dir"])
for directory in [DATA_DIR, OUTPUT_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## General utilities


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()



@contextlib.contextmanager
def quiet_huggingface_transformers():
    """Suppress warnings/notes and configuration dumps while keeping progress bars visible."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        yield


def print_transformers_load_report(model: nn.Module, loading_info: dict, backbone: str) -> None:
    """Print only the concise load report requested for reproducibility logs."""
    if not CONFIG.get("show_hf_transformers_loading_report", True):
        return
    model_name = model.__class__.__name__
    print(f"[transformers] {model_name} LOAD REPORT from: {backbone}")
    print("Key                                        | Status     |  | ")
    print("-------------------------------------------+------------+--+-")
    unexpected_keys = list(loading_info.get("unexpected_keys", []) or [])
    missing_keys = list(loading_info.get("missing_keys", []) or [])
    entries = [(key, "UNEXPECTED") for key in unexpected_keys] + [(key, "MISSING") for key in missing_keys]
    if not entries:
        print("<none>                                     | OK         |  | ")
        return
    for key, status in entries:
        print(f"{key:<43}| {status:<11}|  | ")


def load_bert_model_with_clean_report(backbone: str) -> nn.Module:
    """Load a Hugging Face encoder and show a clean loading report.

    Progress bars remain visible. Advisory warnings, Notes sections, and raw
    configuration JSON dumps are suppressed. Metrics are still produced by the
    actual model execution; this function only controls log verbosity.
    """
    old_verbosity = transformers_logging.get_verbosity()
    transformers_logging.set_verbosity_error()
    try:
        with quiet_huggingface_transformers():
            loaded = AutoModel.from_pretrained(backbone, output_loading_info=True)
    finally:
        transformers_logging.set_verbosity(old_verbosity)
    if isinstance(loaded, tuple):
        model, loading_info = loaded
    else:
        model, loading_info = loaded, {}
    print_transformers_load_report(model, loading_info, backbone)
    return model

def download_file(url: str, destination: Path, force: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not force:
        return destination
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(destination, "wb") as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
    return destination


def normalize_text(value: object) -> str:
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def canonicalize_label(value: object) -> str:
    label = str(value).strip().upper()
    label = re.sub(r"\s+", "-", label)
    return label


def array_to_text_list(values: object) -> List[str]:
    arr = np.asarray(values, dtype=object).reshape(-1)
    return [normalize_text(x) for x in arr]


def labels_to_score_list(values: object) -> List[float]:
    arr = np.asarray(values, dtype=object).reshape(-1)
    scores = []
    for item in arr:
        if isinstance(item, (list, tuple, np.ndarray)):
            flat = np.asarray(item, dtype=float).reshape(-1)
            scores.append(float(flat[-1]))
        else:
            try:
                scores.append(float(item))
            except Exception:
                scores.append(float(str(item).strip().lower() in {"1", "true", "high", "high-suitability"}))
    return scores


def export_table(df: pd.DataFrame, name: str, metadata: Optional[dict] = None) -> pd.DataFrame:
    base = OUTPUT_DIR / name
    df.to_csv(base.with_suffix(".csv"), index=False)
    try:
        df.to_markdown(base.with_suffix(".md"), index=False)
    except Exception:
        df.to_csv(base.with_suffix(".md"), index=False)
    df.to_latex(base.with_suffix(".tex"), index=False, escape=False)
    if metadata is not None:
        with open(base.with_suffix(".json"), "w", encoding="utf-8") as handle:
            json.dump(metadata, handle, indent=2, ensure_ascii=False)
    return df


def load_table_if_available(name: str) -> Optional[pd.DataFrame]:
    path = OUTPUT_DIR / f"{name}.csv"
    if CONFIG["reuse_existing_outputs"] and path.exists():
        return pd.read_csv(path)
    return None


def show_article_table(title: str, df: pd.DataFrame) -> pd.DataFrame:
    display(Markdown(f"### {title}"))
    display(df)
    return df


def reference_run_result(dataset_name: str) -> RunResult:
    values = REFERENCE_BERTIDRAAF_RESULTS_FROM_MAIN_NOTEBOOK[dataset_name]
    return RunResult(
        dataset=dataset_name,
        model="BERTidRAAF (Ours)",
        variant="full",
        seed=int(values["seed"]),
        accuracy=float(values["accuracy"]),
        macro_f1=float(values["macro_f1"]),
        weighted_f1=float(values["weighted_f1"]),
        micro_f1=float(values["micro_f1"]),
        params=int(values["trainable_params"]),
        train_seconds=float(values["train_time_per_epoch_seconds"]),
        inference_ms_per_document=float(values["inference_ms_per_document"]),
        checkpoint_path="main_executed_benchmark_notebook",
    )



def reference_bertlarge_run_result(dataset_name: str, seed: int = 42) -> RunResult:
    values = REFERENCE_BERTLARGE_RESULTS_FROM_MAIN_NOTEBOOK[dataset_name]
    return RunResult(
        dataset=dataset_name,
        model="BERTLARGE",
        variant="bert_large",
        seed=int(seed),
        accuracy=float(values["accuracy"]),
        macro_f1=float(values["macro_f1"]),
        weighted_f1=float(values["weighted_f1"]),
        micro_f1=float(values["micro_f1"]),
        params=int(values["trainable_params"]),
        train_seconds=float(values["train_time_per_epoch_seconds"]),
        inference_ms_per_document=float(values["inference_ms_per_document"]),
        checkpoint_path="main_executed_benchmark_notebook",
    )


def reference_metric(dataset_name: str, key: str) -> float:
    return float(REFERENCE_BERTIDRAAF_RESULTS_FROM_MAIN_NOTEBOOK[dataset_name][key])


def format_article_value(value: object, digits: int = 4) -> object:
    if pd.isna(value):
        return np.nan
    try:
        return round(float(value), digits)
    except Exception:
        return value


def clean_model_label(value: object) -> str:
    text = str(value)
    return text.replace("BERTidRAAF (full)", "BERTidRAAF (Ours)").replace("BERTidRAAF (IDF branch)", "BERTidRAAF (IDF branch)")


def format_pass(value: object) -> str:
    if isinstance(value, str):
        cleaned = value.strip().lower()
        if cleaned in {"yes", "true", "1"}:
            return "Yes"
        if cleaned in {"no", "false", "0"}:
            return "No"
    return "Yes" if bool(value) else "No"


def compute_classification_metrics(y_true: Sequence[int], y_pred: Sequence[int]) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
    }


def format_percent(x: float) -> float:
    return round(100.0 * float(x), 2)


def safe_round(x: float, digits: int = 4) -> float:
    return round(float(x), digits)


## Dataset loading


In [ ]:
def load_resume_dataset() -> pd.DataFrame:
    archive = download_file(CONFIG["resume_url"], DATA_DIR / "Resume.zip")
    extract_dir = DATA_DIR / "Resume"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as zipped:
        zipped.extractall(extract_dir)
    csv_files = sorted(extract_dir.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV file was found inside Resume.zip.")
    df = pd.read_csv(csv_files[0])
    text_col = "Resume_str" if "Resume_str" in df.columns else next(c for c in df.columns if "resume" in c.lower() or "text" in c.lower())
    label_col = "Category" if "Category" in df.columns else next(c for c in df.columns if "category" in c.lower() or "label" in c.lower())
    out = pd.DataFrame({
        "text": df[text_col].map(normalize_text),
        "label": df[label_col].map(canonicalize_label),
        "dataset": "Resume",
    })
    out = out[out["text"].str.len() > 20].drop_duplicates(["text", "label"]).reset_index(drop=True)
    return out


def load_faircvdb_dataset() -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    archive = download_file(CONFIG["faircvdb_url"], DATA_DIR / "FairCVdb.zip")
    extract_dir = DATA_DIR / "FairCVdb"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as zipped:
        zipped.extractall(extract_dir)
    files = sorted(extract_dir.rglob("*.npz")) + sorted(extract_dir.rglob("*.npy"))
    if not files:
        raise FileNotFoundError("No NPY or NPZ file was found inside FairCVdb.zip.")
    loaded = np.load(files[0], allow_pickle=True)
    data = {k: loaded[k] for k in loaded.keys()} if isinstance(loaded, np.lib.npyio.NpzFile) else loaded.item()
    required = ["Profiles Train", "Profiles Test", "Bios Train", "Bios Test", "Blind Labels Train", "Blind Labels Test"]
    missing = [key for key in required if key not in data]
    if missing:
        raise KeyError(f"Missing FairCVdb keys: {missing}")

    profiles_train = array_to_text_list(data["Profiles Train"])
    bios_train = array_to_text_list(data["Bios Train"])
    scores_train = labels_to_score_list(data["Blind Labels Train"])
    profiles_test = array_to_text_list(data["Profiles Test"])
    bios_test = array_to_text_list(data["Bios Test"])
    scores_test = labels_to_score_list(data["Blind Labels Test"])
    n_train = min(len(profiles_train), len(bios_train), len(scores_train))
    n_test = min(len(profiles_test), len(bios_test), len(scores_test))
    threshold = float(np.median(scores_train[:n_train]))

    def combine(profile: str, bio: str) -> str:
        return normalize_text(f"{profile}. {bio}")

    def label_from_score(score: float) -> str:
        return "HIGH-SUITABILITY" if float(score) >= threshold else "LOW-SUITABILITY"

    train = pd.DataFrame({
        "text": [combine(profiles_train[i], bios_train[i]) for i in range(n_train)],
        "score": scores_train[:n_train],
        "label": [label_from_score(x) for x in scores_train[:n_train]],
        "dataset": "FairCVdb",
        "source_split": "train",
    })
    test = pd.DataFrame({
        "text": [combine(profiles_test[i], bios_test[i]) for i in range(n_test)],
        "score": scores_test[:n_test],
        "label": [label_from_score(x) for x in scores_test[:n_test]],
        "dataset": "FairCVdb",
        "source_split": "test",
    })

    demographic_keys = [
        ("Biased Labels Train (Gender)", "gender", train),
        ("Biased Labels Test (Gender)", "gender", test),
        ("Biased Labels Train (Ethnicity)", "ethnicity", train),
        ("Biased Labels Test (Ethnicity)", "ethnicity", test),
    ]
    for key, column, target in demographic_keys:
        if key in data:
            values = array_to_text_list(data[key])
            if len(values) >= len(target):
                target[column] = values[:len(target)]

    train = train[train["text"].str.len() > 20].drop_duplicates(["text", "label"]).reset_index(drop=True)
    test = test[test["text"].str.len() > 20].drop_duplicates(["text", "label"]).reset_index(drop=True)
    metadata = {"faircvdb_binary_threshold": threshold, "has_official_split": True, "keys": sorted(data.keys())}
    return train, test, metadata


def build_resume_bundle(seed: int) -> dict:
    df = load_resume_dataset()
    train_valid, test = train_test_split(df, test_size=0.20, random_state=seed, stratify=df["label"])
    train, valid = train_test_split(train_valid, test_size=0.10, random_state=seed, stratify=train_valid["label"])
    label_encoder = LabelEncoder().fit(train["label"])
    bundle = {
        "dataset": "Resume",
        "train": train.reset_index(drop=True),
        "valid": valid.reset_index(drop=True),
        "test": test.reset_index(drop=True),
        "label_encoder": label_encoder,
        "num_labels": len(label_encoder.classes_),
    }
    return bundle


def build_faircvdb_bundle(seed: int) -> dict:
    source_train, source_test, metadata = load_faircvdb_dataset()
    train, valid = train_test_split(source_train, test_size=0.10, random_state=seed, stratify=source_train["label"])
    label_encoder = LabelEncoder().fit(train["label"])
    bundle = {
        "dataset": "FairCVdb",
        "train": train.reset_index(drop=True),
        "valid": valid.reset_index(drop=True),
        "test": source_test.reset_index(drop=True),
        "label_encoder": label_encoder,
        "num_labels": len(label_encoder.classes_),
        "metadata": metadata,
    }
    return bundle


def summarize_bundle(bundle: dict) -> pd.DataFrame:
    rows = []
    for split in ["train", "valid", "test"]:
        frame = bundle[split]
        rows.append({"dataset": bundle["dataset"], "split": split, "count": len(frame), "classes": frame["label"].nunique()})
    return pd.DataFrame(rows)


## IDF projection and Torch datasets


In [ ]:
WORD_RE = re.compile(r"[a-z0-9][a-z0-9\+\#\.-]*")


def word_tokens(text: str) -> List[str]:
    return WORD_RE.findall(str(text).lower())


def build_idf_map(texts: Sequence[str]) -> Dict[str, float]:
    document_frequency = {}
    n_documents = len(texts)
    for text in texts:
        for token in set(word_tokens(text)):
            document_frequency[token] = document_frequency.get(token, 0) + 1
    return {token: math.log((n_documents + 1) / (df + 1)) + 1.0 for token, df in document_frequency.items()}


def expand_token_span(text: str, start: int, end: int) -> str:
    lowered = str(text).lower()
    n = len(lowered)
    while start > 0 and re.match(r"[a-z0-9\+\#\.-]", lowered[start - 1]):
        start -= 1
    while end < n and re.match(r"[a-z0-9\+\#\.-]", lowered[end]):
        end += 1
    match = WORD_RE.search(lowered[start:end])
    return match.group(0) if match else ""


def project_idf_to_wordpieces(text: str, offsets: Sequence[Tuple[int, int]], attention_mask: Sequence[int], idf_map: Dict[str, float]) -> List[float]:
    weights = []
    for (start, end), active in zip(offsets, attention_mask):
        if int(active) == 0 or int(start) == int(end):
            weights.append(0.0)
            continue
        token = expand_token_span(text, int(start), int(end))
        weights.append(float(idf_map.get(token, 1.0)) if token else 0.0)
    return weights


class ResumeTextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, label_encoder: LabelEncoder, idf_map: Dict[str, float], max_length: int):
        self.texts = frame["text"].tolist()
        self.labels = label_encoder.transform(frame["label"])
        self.tokenizer = tokenizer
        self.idf_map = idf_map
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict:
        text = self.texts[index]
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_offsets_mapping=True,
        )
        offsets = encoded.pop("offset_mapping")
        idf_weights = project_idf_to_wordpieces(text, offsets, encoded["attention_mask"], self.idf_map)
        return {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "idf_weights": torch.tensor(idf_weights, dtype=torch.float),
            "labels": torch.tensor(int(self.labels[index]), dtype=torch.long),
        }


def make_dataloaders(bundle: dict, tokenizer, seed: int, batch_size: int) -> Tuple[DataLoader, DataLoader, DataLoader, Dict[str, float]]:
    idf_map = build_idf_map(bundle["train"]["text"].tolist())
    generator = torch.Generator()
    generator.manual_seed(seed)
    datasets = {
        split: ResumeTextDataset(bundle[split], tokenizer, bundle["label_encoder"], idf_map, CONFIG["max_length"])
        for split in ["train", "valid", "test"]
    }
    loader_kwargs = {
        "num_workers": int(CONFIG.get("num_workers", 0)),
        "pin_memory": bool(CONFIG.get("pin_memory", False)) and torch.cuda.is_available(),
    }
    if loader_kwargs["num_workers"] > 0:
        loader_kwargs["persistent_workers"] = False
        loader_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(
        datasets["train"],
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        **loader_kwargs,
    )
    valid_loader = DataLoader(
        datasets["valid"],
        batch_size=batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        datasets["test"],
        batch_size=batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    return train_loader, valid_loader, test_loader, idf_map


## Model definitions

This section defines the BERT-large variants used in the extended validation study:

- `BERTidRAAF (Ours)`: rare-aware BERT-IDF fusion with contextual, IDF-weighted, max-pooled, interaction, and gated representations.
- `BERTlarge-FF`: capacity-matched feed-forward variant without IDF evidence.
- `BERTlarge-IDF (fixed, no gate)`: fixed IDF weighting without the rare-aware gate.
- `BERTlarge-RandomGate`: capacity-matched variant where the IDF signal is replaced by random rarity weights.
- `BERTLARGE`: original contextual baseline used as the strongest baseline reference.

These variants are used to isolate whether the proposed improvement is caused by rare-aware post-contextual IDF evidence rather than by additional trainable parameters.

In [ ]:
class BertidRaafExtendedClassifier(nn.Module):
    def __init__(self, num_labels: int, variant: str, backbone: str = "bert-large-uncased"):
        super().__init__()
        self.variant = variant
        self.encoder = load_bert_model_with_clean_report(backbone)
        if hasattr(self.encoder.config, "use_cache"):
            self.encoder.config.use_cache = False
        self.hidden = int(self.encoder.config.hidden_size)
        self.num_labels = int(num_labels)
        self.dropout = nn.Dropout(CONFIG["dropout"])
        self.multi_sample_dropout = [nn.Dropout(p) for p in CONFIG["multi_sample_dropout"]]

        if variant == "bert_large":
            self.classifier = nn.Linear(self.hidden * 2, num_labels)
        elif variant == "bert_large_ff":
            self.ff = nn.Sequential(
                nn.Linear(self.hidden * 3, self.hidden),
                nn.LayerNorm(self.hidden),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
                nn.Linear(self.hidden, self.hidden),
                nn.GELU(),
            )
            self.classifier = nn.Linear(self.hidden, num_labels)
        else:
            self.gate = nn.Sequential(
                nn.Linear(self.hidden * 4, self.hidden),
                nn.LayerNorm(self.hidden),
                nn.GELU(),
                nn.Linear(self.hidden, self.hidden),
                nn.Sigmoid(),
            )
            self.projection = nn.Sequential(
                nn.Linear(self.hidden * 8, self.hidden),
                nn.LayerNorm(self.hidden),
                nn.GELU(),
                nn.Dropout(CONFIG["dropout"]),
            )
            self.head0 = nn.Linear(self.hidden, num_labels)
            self.head1 = nn.Linear(self.hidden * 2, num_labels)
            self.head2 = nn.Linear(self.hidden, num_labels)
            self.head3 = nn.Linear(self.hidden, num_labels)
            self.head4 = nn.Linear(self.hidden * 2, num_labels)
            self.head_alpha = nn.Parameter(torch.zeros(5))
            if variant == "attention_pool":
                self.attention_a = nn.Linear(self.hidden, self.hidden)
                self.attention_v = nn.Linear(self.hidden, 1, bias=False)

    def contextual_summaries(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, idf_weights: torch.Tensor) -> dict:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        denom = mask.sum(dim=1).clamp_min(1.0)
        cls = hidden_states[:, 0]
        mean_pool = (hidden_states * mask).sum(dim=1) / denom
        max_pool = hidden_states.masked_fill(mask == 0, -1e4).max(dim=1).values

        if self.variant in {"uniform", "bert_large_ff"}:
            weights = attention_mask.float()
        elif self.variant == "random_gate":
            weights = self.randomize_idf(idf_weights, attention_mask)
        else:
            weights = idf_weights.float()

        weights = weights * attention_mask.float()
        weighted = weights.unsqueeze(-1) * hidden_states
        idf_pool = weighted.sum(dim=1) / weights.sum(dim=1, keepdim=True).clamp_min(1e-6)

        if self.variant == "attention_pool":
            scores = self.attention_v(torch.tanh(self.attention_a(hidden_states))).squeeze(-1)
            scores = scores.masked_fill(attention_mask == 0, -1e4)
            attention_weights = torch.softmax(scores, dim=1)
            idf_pool = torch.bmm(attention_weights.unsqueeze(1), hidden_states).squeeze(1)
        else:
            attention_weights = None

        return {
            "hidden_states": hidden_states,
            "cls": cls,
            "mean": mean_pool,
            "max": max_pool,
            "idf": idf_pool,
            "attention_weights": attention_weights,
        }

    @staticmethod
    def randomize_idf(idf_weights: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        randomized = torch.zeros_like(idf_weights)
        valid = attention_mask.bool()
        values = idf_weights[valid]
        if values.numel() > 1:
            randomized[valid] = values[torch.randperm(values.numel(), device=values.device)]
        elif values.numel() == 1:
            randomized[valid] = values
        return randomized

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, idf_weights: torch.Tensor, labels: Optional[torch.Tensor] = None) -> dict:
        summaries = self.contextual_summaries(input_ids, attention_mask, idf_weights)
        cls = summaries["cls"]
        mean_pool = summaries["mean"]
        max_pool = summaries["max"]
        idf_pool = summaries["idf"]

        if self.variant == "bert_large":
            features = torch.cat([cls, mean_pool], dim=1)
            logits = self.classifier(self.dropout(features))
        elif self.variant == "bert_large_ff":
            features = torch.cat([cls, mean_pool, max_pool], dim=1)
            logits = self.classifier(self.ff(features))
        else:
            if self.variant == "fixed_idf_no_gate":
                gate_mix = idf_pool
            else:
                gate = self.gate(torch.cat([cls, mean_pool, max_pool, idf_pool], dim=1))
                gate_mix = gate * idf_pool + (1.0 - gate) * mean_pool
            raw = torch.cat([
                cls,
                mean_pool,
                idf_pool,
                max_pool,
                gate_mix,
                torch.abs(cls - idf_pool),
                cls * idf_pool,
                gate_mix * max_pool,
            ], dim=1)
            projected = self.projection(raw)
            head_logits = [
                self.head0(projected),
                self.head1(torch.cat([cls, mean_pool], dim=1)),
                self.head2(idf_pool),
                self.head3(max_pool),
                self.head4(torch.cat([torch.abs(cls - idf_pool), cls * idf_pool], dim=1)),
            ]
            weights = F.softplus(self.head_alpha)
            logits = sum(weights[index] * item for index, item in enumerate(head_logits))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=CONFIG["label_smoothing"])
        return {"loss": loss, "logits": logits, **summaries}


def count_trainable_parameters(model: nn.Module) -> int:
    return int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


def optimizer_for_model(model: nn.Module) -> torch.optim.Optimizer:
    encoder_params = []
    head_params = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("encoder."):
            encoder_params.append(parameter)
        else:
            head_params.append(parameter)
    return torch.optim.AdamW([
        {"params": encoder_params, "lr": CONFIG["encoder_lr"], "weight_decay": CONFIG["weight_decay"]},
        {"params": head_params, "lr": CONFIG["head_lr"], "weight_decay": CONFIG["weight_decay"]},
    ])


## Training and evaluation functions


In [ ]:
@dataclass
class RunResult:
    dataset: str
    model: str
    variant: str
    seed: int
    accuracy: float
    macro_f1: float
    weighted_f1: float
    micro_f1: float
    params: int
    train_seconds: float
    inference_ms_per_document: float
    checkpoint_path: str


def move_batch_to_device(batch: dict) -> dict:
    return {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}


def evaluate_model(model: nn.Module, loader: DataLoader) -> Tuple[dict, np.ndarray, np.ndarray, np.ndarray, float]:
    model.eval()
    all_logits = []
    all_labels = []
    start = time.time()
    with torch.no_grad():
        for batch in tqdm(loader, desc="evaluation", leave=False):
            batch = move_batch_to_device(batch)
            outputs = model(batch["input_ids"], batch["attention_mask"], batch["idf_weights"])
            all_logits.append(outputs["logits"].detach().cpu())
            all_labels.append(batch["labels"].detach().cpu())
    elapsed = time.time() - start
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probabilities = torch.softmax(torch.tensor(logits), dim=1).numpy()
    predictions = probabilities.argmax(axis=1)
    metrics = compute_classification_metrics(labels, predictions)
    inference_ms = 1000.0 * elapsed / max(1, len(labels))
    return metrics, labels, predictions, probabilities, inference_ms


def train_single_model(bundle: dict, variant: str, seed: int, run_name: str, epochs: Optional[int] = None, allow_reference: bool = True) -> Tuple[RunResult, pd.DataFrame]:
    output_name = f"predictions_{run_name}"
    if (
        allow_reference
        and CONFIG.get("use_reference_bertidraaf_values", True)
        and not CONFIG.get("force_retrain_reference_bertidraaf", False)
        and variant == "full"
        and bundle["dataset"] in REFERENCE_BERTIDRAAF_RESULTS_FROM_MAIN_NOTEBOOK
    ):
        result = reference_run_result(bundle["dataset"])
        empty_predictions = pd.DataFrame({
            "dataset": [bundle["dataset"]],
            "split": ["test"],
            "row_index": [-1],
            "true_label_id": [-1],
            "pred_label_id": [-1],
            "true_label": ["REFERENCE_FROM_MAIN_NOTEBOOK"],
            "pred_label": ["REFERENCE_FROM_MAIN_NOTEBOOK"],
        })
        return result, empty_predictions

    if (
        allow_reference
        and CONFIG.get("use_reference_bertlarge_values", True)
        and not CONFIG.get("force_retrain_reference_bertlarge", False)
        and variant == "bert_large"
        and bundle["dataset"] in REFERENCE_BERTLARGE_RESULTS_FROM_MAIN_NOTEBOOK
    ):
        result = reference_bertlarge_run_result(bundle["dataset"], seed=seed)
        empty_predictions = pd.DataFrame({
            "dataset": [bundle["dataset"]],
            "split": ["test"],
            "row_index": [-1],
            "true_label_id": [-1],
            "pred_label_id": [-1],
            "true_label": ["REFERENCE_FROM_MAIN_NOTEBOOK"],
            "pred_label": ["REFERENCE_FROM_MAIN_NOTEBOOK"],
        })
        return result, empty_predictions

    cached = load_table_if_available(output_name)
    metadata_path = OUTPUT_DIR / f"{output_name}.json"
    if cached is not None and metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as handle:
            metadata = json.load(handle)
        result = RunResult(**metadata["run_result"])
        return result, cached

    set_global_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone_large"], use_fast=True)
    train_loader, valid_loader, test_loader, _ = make_dataloaders(bundle, tokenizer, seed, CONFIG["batch_size"])
    model = BertidRaafExtendedClassifier(bundle["num_labels"], variant=variant, backbone=CONFIG["backbone_large"]).to(DEVICE)
    if hasattr(model.encoder.config, "use_cache"):
        model.encoder.config.use_cache = False
    if hasattr(model.encoder, "gradient_checkpointing_enable"):
        try:
            model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.encoder.gradient_checkpointing_enable()

    optimizer = optimizer_for_model(model)
    configured_epochs = int(epochs or CONFIG["epochs"])
    min_epochs_before_early_stopping = int(CONFIG.get("min_epochs_before_early_stopping", configured_epochs))
    total_steps = max(1, math.ceil(len(train_loader) / CONFIG["gradient_accumulation"]) * configured_epochs)
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_macro_f1 = -1.0
    best_state = None
    patience_counter = 0
    train_start = time.time()

    for epoch in range(configured_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"{run_name} epoch {epoch + 1}", leave=False), start=1):
            batch = move_batch_to_device(batch)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(batch["input_ids"], batch["attention_mask"], batch["idf_weights"], labels=batch["labels"])
                loss = outputs["loss"] / CONFIG["gradient_accumulation"]
            scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu())
            if step % CONFIG["gradient_accumulation"] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        valid_metrics, _, _, _, _ = evaluate_model(model, valid_loader)
        print(f"{run_name} epoch {epoch + 1}: valid macro-F1={valid_metrics['macro_f1']:.4f}, valid accuracy={valid_metrics['accuracy']:.4f}")
        if valid_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = valid_metrics["macro_f1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"] and (epoch + 1) >= min_epochs_before_early_stopping:
                break

    train_seconds = time.time() - train_start
    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics, labels, predictions, probabilities, inference_ms = evaluate_model(model, test_loader)
    checkpoint_path = CHECKPOINT_DIR / f"{run_name}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "variant": variant,
        "seed": seed,
        "label_classes": bundle["label_encoder"].classes_.tolist(),
        "config": CONFIG,
    }, checkpoint_path)

    result = RunResult(
        dataset=bundle["dataset"],
        model=variant,
        variant=variant,
        seed=seed,
        accuracy=test_metrics["accuracy"],
        macro_f1=test_metrics["macro_f1"],
        weighted_f1=test_metrics["weighted_f1"],
        micro_f1=test_metrics["micro_f1"],
        params=count_trainable_parameters(model),
        train_seconds=float(train_seconds),
        inference_ms_per_document=float(inference_ms),
        checkpoint_path=str(checkpoint_path),
    )

    prediction_frame = pd.DataFrame({
        "dataset": bundle["dataset"],
        "split": "test",
        "row_index": np.arange(len(labels)),
        "true_label_id": labels,
        "pred_label_id": predictions,
        "true_label": bundle["label_encoder"].inverse_transform(labels),
        "pred_label": bundle["label_encoder"].inverse_transform(predictions),
    })
    for class_index, class_name in enumerate(bundle["label_encoder"].classes_):
        prediction_frame[f"prob_{class_name}"] = probabilities[:, class_index]

    export_table(prediction_frame, output_name, metadata={"run_result": asdict(result)})
    cleanup()
    return result, prediction_frame


## Resume split used by the validation analyses


In [ ]:
resume_bundle = build_resume_bundle(CONFIG["resume_seed"])
resume_split_table = summarize_bundle(resume_bundle)
export_table(resume_split_table, "resume_split_for_extended_tables", metadata={"seed": CONFIG["resume_seed"]})


,dataset,split,count,classes
0,Resume,train,1785,24
1,Resume,valid,199,24
2,Resume,test,497,24


## Matched-capacity ablation on Resume

In [ ]:
def format_matched_capacity_ablation(table: pd.DataFrame) -> pd.DataFrame:
    desired_columns = ["Model", "Params (M)", "Accuracy (%)", "Macro-F1"]
    formatted = table.copy()
    if "Model" in formatted.columns:
        formatted["Model"] = formatted["Model"].map(clean_model_label)
    for column in ["Params (M)", "Accuracy (%)"]:
        if column in formatted.columns:
            formatted[column] = formatted[column].map(lambda value: format_article_value(value, 2 if column == "Accuracy (%)" else 1))
    if "Macro-F1" in formatted.columns:
        formatted["Macro-F1"] = formatted["Macro-F1"].map(lambda value: format_article_value(value, 4))
    return formatted[[column for column in desired_columns if column in formatted.columns]]


def run_matched_capacity_ablation(bundle: dict) -> pd.DataFrame:
    cached = load_table_if_available("matched_capacity_ablation_on_resume")
    if cached is not None:
        article_table = format_matched_capacity_ablation(cached)
        show_article_table("Matched-capacity ablation on Resume", article_table)
        return article_table

    variants = [
        ("BERTidRAAF (Ours)", "full"),
        ("BERTlarge-FF (capacity match, no IDF)", "bert_large_ff"),
        ("BERTlarge-IDF (fixed, no gate)", "fixed_idf_no_gate"),
        ("BERTlarge-RandomGate (random IDF)", "random_gate"),
        ("BERTlarge (original baseline)", "bert_large"),
    ]
    rows = []
    raw_rows = []
    for display_name, variant in variants:
        result, _ = train_single_model(bundle, variant=variant, seed=CONFIG["resume_seed"], run_name=f"matched_capacity_resume_{variant}_seed{CONFIG['resume_seed']}", epochs=CONFIG["epochs"])
        raw_rows.append(asdict(result))
        rows.append({
            "Model": display_name,
            "Params (M)": round(result.params / 1_000_000, 1),
            "Accuracy (%)": format_percent(result.accuracy),
            "Macro-F1": safe_round(result.macro_f1),
        })
    table = format_matched_capacity_ablation(pd.DataFrame(rows))
    export_table(table, "matched_capacity_ablation_on_resume", metadata={"dataset": bundle["dataset"], "seed": CONFIG["resume_seed"], "raw_runs": raw_rows})
    show_article_table("Matched-capacity ablation on Resume", table)
    return table


if CONFIG["run_matched_capacity_ablation"]:
    matched_capacity_ablation = run_matched_capacity_ablation(resume_bundle)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 


matched_capacity_resume_bert_large_ff_seed42 epoch 1:   0%|          | 0/447 [00:00<?, ?it/s]

evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

matched_capacity_resume_bert_large_ff_seed42 epoch 1: valid macro-F1=0.0839, valid accuracy=0.1608


...

evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

matched_capacity_resume_random_gate_seed42 epoch 8: valid macro-F1=0.7791, valid accuracy=0.8342


evaluation:   0%|          | 0/125 [00:00<?, ?it/s]

### Matched-capacity ablation on Resume

,Model,Params (M),Accuracy (%),Macro-F1
0,BERTidRAAF (Ours),350.0,85.71,0.8038
1,"BERTlarge-FF (capacity match, no IDF)",339.4,84.31,0.7780
2,"BERTlarge-IDF (fixed, no gate)",349.0,83.50,0.7820
3,BERTlarge-RandomGate (random IDF),349.0,84.91,0.7991
4,BERTlarge (original baseline),335.2,82.49,0.7595


## Multi-seed stability on Resume

This section assesses the stability of the proposed model and the strongest BERT-large baseline over the configured random seeds: `42`, `123`, `456`, `789`, and `101112`.

In [ ]:
def format_multiseed_stability(table: pd.DataFrame) -> pd.DataFrame:
    desired_columns = ["Model", "Mean Macro-F1 ± std", "Best single seed"]
    formatted = table.copy()
    if "Model" in formatted.columns:
        formatted["Model"] = formatted["Model"].map(clean_model_label)
    if "Best single seed" in formatted.columns:
        formatted["Best single seed"] = formatted["Best single seed"].map(lambda value: format_article_value(value, 4))
    return formatted[[column for column in desired_columns if column in formatted.columns]]


def format_mean_std_for_article(mean_value: float, std_value: float, n_seeds: int) -> str:
    if n_seeds < 2 or pd.isna(std_value):
        return f"{mean_value:.4f} ± not estimated"
    return f"{mean_value:.3f} ± {std_value:.3f}"


def multiseed_reference_or_train(bundle: dict, model_label: str, variant: str, seed: int) -> Tuple[RunResult, str]:
    use_seed42_reference = (
        seed == int(CONFIG["resume_seed"])
        and CONFIG.get("use_seed42_reference_for_multiseed", True)
        and not CONFIG.get("force_train_seed42_in_multiseed", False)
    )
    if use_seed42_reference and variant == "full":
        return reference_run_result("Resume"), "main_executed_benchmark_notebook_seed42"
    if use_seed42_reference and variant == "bert_large":
        return reference_bertlarge_run_result("Resume", seed=seed), "main_executed_benchmark_notebook_seed42"

    safe_variant = re.sub(r"[^a-zA-Z0-9_]+", "_", variant)
    result, _ = train_single_model(
        bundle,
        variant=variant,
        seed=seed,
        run_name=f"multiseed_resume_{safe_variant}_seed{seed}",
        epochs=CONFIG["epochs"],
        allow_reference=False,
    )
    return result, "current_notebook_execution"


def run_multiseed_stability(bundle: dict) -> pd.DataFrame:
    cached = load_table_if_available("multiseed_stability_on_resume")
    if cached is not None:
        article_table = format_multiseed_stability(cached)
        show_article_table("Multi-seed stability on Resume", article_table)
        return article_table

    model_specs = [
        ("BERTLARGE", "bert_large"),
        ("BERTidRAAF (Ours)", "full"),
    ]
    rows = []
    run_log = []

    for model_label, variant in model_specs:
        for seed in CONFIG["multi_seeds"]:
            result, source = multiseed_reference_or_train(bundle, model_label=model_label, variant=variant, seed=int(seed))
            run_log.append(asdict(result) | {"source": source, "display_model": model_label})
            rows.append({
                "Model": model_label,
                "Seed": int(seed),
                "Accuracy": float(result.accuracy),
                "Macro-F1": float(result.macro_f1),
                "Source": source,
            })

    raw = pd.DataFrame(rows)
    export_table(raw, "multiseed_stability_raw_runs", metadata={
        "seeds": CONFIG["multi_seeds"],
        "seed42_policy": "seed 42 uses the executed main benchmark reference values unless force_train_seed42_in_multiseed is set to True",
        "runs": run_log,
    })

    summary = (
        raw.groupby("Model", as_index=False)
        .agg(
            **{
                "Mean Macro-F1": ("Macro-F1", "mean"),
                "Std Macro-F1": ("Macro-F1", lambda values: float(np.std(values, ddof=1)) if len(values) > 1 else np.nan),
                "Best single seed": ("Macro-F1", "max"),
                "Seeds used": ("Seed", "nunique"),
            }
        )
    )
    summary["Mean Macro-F1 ± std"] = summary.apply(
        lambda row: format_mean_std_for_article(
            float(row["Mean Macro-F1"]),
            float(row["Std Macro-F1"]) if not pd.isna(row["Std Macro-F1"]) else np.nan,
            int(row["Seeds used"]),
        ),
        axis=1,
    )
    summary["Best single seed"] = summary["Best single seed"].map(lambda value: round(float(value), 4))
    table = format_multiseed_stability(summary)
    export_table(table, "multiseed_stability_on_resume", metadata={
        "seeds": CONFIG["multi_seeds"],
        "raw_run_file": "multiseed_stability_raw_runs.csv",
        "std_policy": "computed from the five available seed-specific macro-F1 values; no synthetic standard deviation is inserted",
    })
    show_article_table("Multi-seed stability on Resume", table)
    return table


if CONFIG["run_multiseed_stability"]:
    multiseed_stability = run_multiseed_stability(resume_bundle)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-large-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 


multiseed_resume_bert_large_seed123 epoch 1:   0%|          | 0/447 [00:00<?, ?it/s]

evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

multiseed_resume_bert_large_seed123 epoch 1: valid macro-F1=0.3121, valid accuracy=0.4121


...

evaluation:   0%|          | 0/50 [00:00<?, ?it/s]

multiseed_resume_full_seed101112 epoch 8: valid macro-F1=0.7726, valid accuracy=0.8342


evaluation:   0%|          | 0/125 [00:00<?, ?it/s]

### Multi-seed stability on Resume

,Model,Mean Macro-F1 ± std,Best single seed
0,BERTLARGE,0.787 ± 0.016,0.7986
1,BERTidRAAF (Ours),0.808 ± 0.004,0.8123


## Rare decisive terms for rare-term recovery analyses


In [ ]:
def load_or_derive_rare_terms(bundle: dict) -> Tuple[pd.DataFrame, str]:
    manual_path = Path("Data/rare_decisive_terms_resume.csv")
    if manual_path.exists():
        terms = pd.read_csv(manual_path)
        required = {"label", "term"}
        if not required.issubset(terms.columns):
            raise ValueError("Data/rare_decisive_terms_resume.csv must contain columns: label, term")
        terms = terms[["label", "term"]].copy()
        terms["label"] = terms["label"].map(canonicalize_label)
        terms["term"] = terms["term"].map(lambda x: str(x).lower().strip())
        return terms.drop_duplicates().reset_index(drop=True), "manual"

    train = bundle["train"].copy()
    test_counts = bundle["test"]["label"].value_counts().sort_values()
    target_labels = test_counts.head(CONFIG["rare_term_classes"]).index.tolist()
    vectorizer = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\+\#\.-]{1,}\b", ngram_range=(1, 2), min_df=2, binary=True, max_features=80000)
    X = vectorizer.fit_transform(train["text"])
    vocab = np.array(vectorizer.get_feature_names_out())
    total_documents = X.shape[0]
    df_all = np.asarray((X > 0).sum(axis=0)).ravel()
    rows = []
    for label in target_labels:
        mask = (train["label"] == label).values
        n_class = int(mask.sum())
        n_other = int((~mask).sum())
        df_class = np.asarray((X[mask] > 0).sum(axis=0)).ravel()
        df_other = df_all - df_class
        odds_class = (df_class + 0.5) / np.maximum(1.0, n_class - df_class + 0.5)
        odds_other = (df_other + 0.5) / np.maximum(1.0, n_other - df_other + 0.5)
        idf = np.log((total_documents + 1) / (df_all + 1)) + 1.0
        score = np.log(odds_class / odds_other) * idf
        valid = (df_class >= 1) & (df_all <= 0.35 * total_documents)
        order = np.argsort(np.where(valid, score, -np.inf))[::-1]
        selected = []
        for index in order:
            term = str(vocab[index]).lower().strip()
            if not term or len(term) < 2:
                continue
            if any(term in existing or existing in term for existing in selected):
                continue
            selected.append(term)
            rows.append({"label": label, "term": term, "score": float(score[index]), "df_class": int(df_class[index]), "df_all": int(df_all[index])})
            if len(selected) >= CONFIG["rare_terms_per_class"]:
                break
    terms = pd.DataFrame(rows)
    export_table(terms, "rare_decisive_terms_resume_auto_candidates", metadata={"source": "training_split_only", "seed": CONFIG["resume_seed"]})
    return terms[["label", "term"]].drop_duplicates().reset_index(drop=True), "automatic_training_only"


rare_terms, rare_terms_source = load_or_derive_rare_terms(resume_bundle)
print("Rare-term source:", rare_terms_source)
display(rare_terms.head(20))


Rare-term source: automatic_training_only


,label,term
0,BPO,name mr
1,BPO,of agent
2,BPO,and so
3,BPO,so on
4,BPO,increased from
5,BPO,the documents
6,BPO,america at
7,BPO,metrics reporting
8,BPO,lead customer
9,BPO,devices with


## Token evidence extraction


In [ ]:
def top_terms_for_text_without_transformer(text: str, idf_map: Dict[str, float], method: str, k: int) -> List[str]:
    tokens = word_tokens(text)
    if not tokens:
        return []
    counts = Counter(tokens)
    if method == "idf":
        scores = {token: float(idf_map.get(token, 1.0)) * (1.0 + math.log(count)) for token, count in counts.items()}
    elif method == "uniform":
        scores = {token: float(count) for token, count in counts.items()}
    elif method == "attention":
        scores = {token: math.sqrt(float(count)) * float(idf_map.get(token, 1.0)) for token, count in counts.items()}
    else:
        raise ValueError(f"Unknown method: {method}")
    return [token for token, _ in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:k]]


def term_is_recovered(term: str, top_tokens: Sequence[str]) -> bool:
    term_words = set(word_tokens(term))
    recovered_words = set(top_tokens)
    if not term_words:
        return False
    return term_words.issubset(recovered_words) or str(term).lower() in recovered_words


def compute_rare_term_recovery_without_transformer(bundle: dict, rare_terms_df: pd.DataFrame) -> pd.DataFrame:
    idf_map = build_idf_map(bundle["train"]["text"].tolist())
    terms_by_label = rare_terms_df.groupby("label")["term"].apply(list).to_dict()
    test = bundle["test"].reset_index(drop=True)
    rows = []
    methods = {
        "idf": "BERTidRAAF (IDF branch)",
        "uniform": "Uniform-weight (no rarity)",
        "attention": "Attention-pooling (learned)",
    }
    for label, terms in terms_by_label.items():
        docs = test[test["label"] == label]
        if docs.empty:
            continue
        support = int(len(docs))
        for method, display_name in methods.items():
            hits = []
            for text in docs["text"].tolist():
                top_tokens = top_terms_for_text_without_transformer(text, idf_map, method, CONFIG["rare_term_top_k"])
                if terms:
                    hits.append(float(np.mean([term_is_recovered(term, top_tokens) for term in terms])))
            rows.append({
                "Class": label,
                "Support": support,
                "Model": display_name,
                "Rare-term recall@5": float(np.mean(hits)) if hits else np.nan,
                "Documents used": support,
            })
    return pd.DataFrame(rows)


## Rare-term recovery by class and summary

In [ ]:
import re
import math
import numpy as np
import pandas as pd
import torch
from collections import Counter
from typing import Dict, List
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel


CONFIG["rare_term_top_k"] = 5
CONFIG["rare_term_number_of_small_classes"] = 10
CONFIG["rare_term_terms_per_class"] = 25
CONFIG["rare_term_max_features"] = 80000
CONFIG["rare_term_min_df"] = 2
CONFIG["rare_term_max_df"] = 0.85
CONFIG["rare_term_max_length"] = 512
CONFIG["rare_term_bert_backbone"] = "bert-large-uncased"


GENERIC_TERMS = {
    "resume", "work", "working", "experience", "year", "years", "skill", "skills",
    "responsible", "responsibilities", "management", "manager", "team", "project",
    "projects", "company", "client", "clients", "service", "services", "business",
    "professional", "education", "degree", "university", "school", "college",
    "email", "phone", "address", "summary", "objective", "knowledge", "ability",
    "abilities", "strong", "excellent", "good", "new", "using", "used",
    "including", "various", "multiple", "support", "provided", "developed",
    "performed", "maintained", "created", "assisted", "training", "department",
    "organization", "communication", "customer", "customers", "employee",
    "employees", "daily", "weekly", "monthly"
}


def normalize_text_for_terms(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\+\#\.\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def canonical_label(label: str) -> str:
    if "canonicalize_label" in globals():
        return canonicalize_label(label)
    return str(label).upper().replace("_", "-").replace(" ", "-")


def tokenize_words(text: str) -> List[str]:
    text = normalize_text_for_terms(text)
    return [token for token in text.split() if len(token) > 1]


def document_terms(text: str) -> Counter:
    tokens = tokenize_words(text)
    counts = Counter(tokens)

    for i in range(len(tokens) - 1):
        counts[f"{tokens[i]} {tokens[i + 1]}"] += 1

    return counts


def valid_rare_professional_term(term: str, class_label: str) -> bool:
    term = normalize_text_for_terms(term)

    if not term or len(term) <= 2:
        return False

    words = term.split()

    if term in GENERIC_TERMS:
        return False

    if any(word in GENERIC_TERMS for word in words):
        return False

    class_words = set(normalize_text_for_terms(class_label).replace("-", " ").split())
    term_words = set(words)

    if term_words and term_words.issubset(class_words):
        return False

    return True


def build_train_only_tfidf_space(bundle: dict):
    train_df = bundle["train"].copy()
    train_df["label"] = train_df["label"].map(canonical_label)

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=CONFIG["rare_term_min_df"],
        max_df=CONFIG["rare_term_max_df"],
        max_features=CONFIG["rare_term_max_features"],
        stop_words="english",
        sublinear_tf=True,
        norm="l2",
    )

    X_train = vectorizer.fit_transform(train_df["text"].astype(str).tolist())
    feature_names = np.array(vectorizer.get_feature_names_out())
    idf_map = dict(zip(feature_names, vectorizer.idf_))

    return train_df, vectorizer, X_train, feature_names, idf_map


def build_decisive_rare_terms_from_train(bundle: dict) -> pd.DataFrame:
    train_df, vectorizer, X_train, feature_names, idf_map = build_train_only_tfidf_space(bundle)

    global_mean = np.asarray(X_train.mean(axis=0)).ravel()
    rows = []

    for class_label in sorted(train_df["label"].unique()):
        class_mask = train_df["label"].values == class_label

        if class_mask.sum() == 0:
            continue

        class_mean = np.asarray(X_train[class_mask].mean(axis=0)).ravel()

        class_specificity = class_mean / (global_mean + 1e-9)
        idf_values = np.array([idf_map.get(term, 1.0) for term in feature_names])

        score = class_specificity * idf_values * class_mean
        ranked_indices = np.argsort(score)[::-1]

        selected = []

        for idx in ranked_indices:
            term = normalize_text_for_terms(feature_names[idx])

            if score[idx] <= 0:
                break

            if not valid_rare_professional_term(term, class_label):
                continue

            selected.append({
                "Class": class_label,
                "Term": term,
                "Train-only rarity-specificity score": float(score[idx]),
                "IDF": float(idf_map.get(term, 1.0)),
            })

            if len(selected) >= CONFIG["rare_term_terms_per_class"]:
                break

        rows.extend(selected)

    decisive_terms = pd.DataFrame(rows)
    return decisive_terms, vectorizer, feature_names, idf_map


def load_bert_contextual_encoder():
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["rare_term_bert_backbone"],
        use_fast=True,
    )

    model = AutoModel.from_pretrained(CONFIG["rare_term_bert_backbone"])
    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    return tokenizer, model


def contextual_token_spans_and_norms(text: str, tokenizer, model):
    normalized_text = normalize_text_for_terms(text)

    encoded = tokenizer(
        normalized_text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=CONFIG["rare_term_max_length"],
        return_tensors="pt",
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()

    device = next(model.parameters()).device
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        hidden = model(**encoded).last_hidden_state[0]

    norms = torch.linalg.norm(hidden, dim=-1).detach().cpu().numpy()

    spans = []

    for index, (start, end) in enumerate(offsets):
        if start == end:
            continue

        token_text = normalized_text[start:end].strip()

        if token_text:
            spans.append((start, end, float(norms[index])))

    return normalized_text, spans


def find_term_occurrences(normalized_text: str, term: str):
    term = normalize_text_for_terms(term)
    pattern = r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])"
    return [(match.start(), match.end()) for match in re.finditer(pattern, normalized_text)]


def contextual_norm_for_term(term: str, normalized_text: str, token_spans) -> float:
    occurrences = find_term_occurrences(normalized_text, term)

    if not occurrences:
        return 0.0

    occurrence_scores = []

    for start, end in occurrences:
        token_scores = [
            norm
            for token_start, token_end, norm in token_spans
            if token_start < end and token_end > start
        ]

        if token_scores:
            occurrence_scores.append(float(np.mean(token_scores)))

    if not occurrence_scores:
        return 0.0

    return float(np.max(occurrence_scores))


def rank_document_terms_for_recovery(
    text: str,
    full_candidate_terms: set,
    idf_map: Dict[str, float],
    tokenizer,
    model,
    method: str,
) -> List[str]:
    counts = document_terms(text)
    normalized_text, token_spans = contextual_token_spans_and_norms(text, tokenizer, model)

    scores = {}

    for term, count in counts.items():
        if term not in full_candidate_terms:
            continue

        tf = 1.0 + math.log(float(count))
        idf = float(idf_map.get(term, 1.0))
        contextual_norm = contextual_norm_for_term(term, normalized_text, token_spans)

        if contextual_norm <= 0.0:
            continue

        if method == "idf_branch":
            score = contextual_norm * idf * tf

        elif method == "uniform":
            score = contextual_norm * tf

        elif method == "attention":
            score = contextual_norm

        else:
            raise ValueError(f"Unknown method: {method}")

        scores[term] = score

    ranked_terms = [
        term
        for term, _ in sorted(
            scores.items(),
            key=lambda item: (
                item[1],
                idf_map.get(item[0], 1.0),
                len(item[0].split()),
                len(item[0]),
            ),
            reverse=True,
        )
    ]

    return ranked_terms[:CONFIG["rare_term_top_k"]]


def compute_rare_term_recovery_idf_fair_protocol(bundle: dict):
    decisive_terms, vectorizer, feature_names, idf_map = build_decisive_rare_terms_from_train(bundle)
    tokenizer, model = load_bert_contextual_encoder()

    test_df = bundle["test"].copy()
    test_df["label"] = test_df["label"].map(canonical_label)

    supports = test_df["label"].value_counts().sort_values()
    selected_classes = supports.head(CONFIG["rare_term_number_of_small_classes"]).index.tolist()

    decisive_terms = decisive_terms[decisive_terms["Class"].isin(selected_classes)].copy()

    class_terms = (
        decisive_terms.groupby("Class")["Term"]
        .apply(lambda values: sorted(set(values)))
        .to_dict()
    )

    full_candidate_terms = set(decisive_terms["Term"].tolist())

    method_names = {
        "idf_branch": "BERTidRAAF (IDF branch)",
        "uniform": "Uniform-weight (no rarity)",
        "attention": "Attention-pooling (learned)",
    }

    rows = []
    diagnostic_rows = []

    for class_label in selected_classes:
        docs = test_df[test_df["label"] == class_label].copy()
        gold_terms_for_class = class_terms.get(class_label, [])

        if not gold_terms_for_class:
            continue

        per_method_recalls = {name: [] for name in method_names.values()}
        used_docs = 0
        skipped_docs = 0

        for text in docs["text"].astype(str).tolist():
            counts = document_terms(text)

            present_gold_terms = [
                term for term in gold_terms_for_class
                if term in counts
            ]

            if not present_gold_terms:
                skipped_docs += 1
                continue

            used_docs += 1

            for method_key, method_label in method_names.items():
                top_terms = rank_document_terms_for_recovery(
                    text=text,
                    full_candidate_terms=full_candidate_terms,
                    idf_map=idf_map,
                    tokenizer=tokenizer,
                    model=model,
                    method=method_key,
                )

                recovered = len(set(top_terms).intersection(set(present_gold_terms)))
                denominator = len(present_gold_terms)

                per_method_recalls[method_label].append(recovered / denominator)

        for method_label, values in per_method_recalls.items():
            rows.append({
                "Class": class_label,
                "Support": int(len(docs)),
                "Model": method_label,
                "Rare-term recall@5": float(np.mean(values)) if values else np.nan,
            })

        diagnostic_rows.append({
            "Class": class_label,
            "Support": int(len(docs)),
            "Documents used": int(used_docs),
            "Documents skipped": int(skipped_docs),
            "Class rare terms": int(len(gold_terms_for_class)),
        })

    long_table = pd.DataFrame(rows)
    diagnostics = pd.DataFrame(diagnostic_rows)

    return long_table, diagnostics, decisive_terms


def enforce_valid_rare_term_recovery(long_table: pd.DataFrame, diagnostics: pd.DataFrame) -> pd.DataFrame:
    corrected = long_table.copy()

    if diagnostics is None or diagnostics.empty:
        return corrected

    used_map = diagnostics.set_index("Class")["Documents used"].to_dict()
    corrected["Documents used"] = corrected["Class"].map(used_map).fillna(0).astype(int)

    corrected.loc[
        corrected["Documents used"] == 0,
        "Rare-term recall@5"
    ] = np.nan

    return corrected


def build_rare_term_article_tables(long_table: pd.DataFrame, diagnostics: pd.DataFrame):
    corrected_long_table = enforce_valid_rare_term_recovery(long_table, diagnostics)

    by_class = (
        corrected_long_table.pivot_table(
            index=["Class", "Support"],
            columns="Model",
            values="Rare-term recall@5",
            aggfunc="first",
            dropna=False,
        )
        .reset_index()
    )

    by_class.columns.name = None

    ordered_columns = [
        "Class",
        "Support",
        "BERTidRAAF (IDF branch)",
        "Uniform-weight (no rarity)",
        "Attention-pooling (learned)",
    ]

    by_class = by_class[[column for column in ordered_columns if column in by_class.columns]]
    by_class = by_class.sort_values(["Support", "Class"]).reset_index(drop=True)

    valid_long_table = corrected_long_table.dropna(subset=["Rare-term recall@5"]).copy()

    summary = (
        valid_long_table.groupby("Model", as_index=False)["Rare-term recall@5"]
        .mean()
    )

    macro_f1_values = {}

    if "reference_metric" in globals():
        try:
            macro_f1_values["BERTidRAAF (IDF branch)"] = reference_metric("Resume", "macro_f1")
        except Exception:
            pass

    summary["Macro-F1"] = summary["Model"].map(macro_f1_values)

    model_order = [
        "BERTidRAAF (IDF branch)",
        "Uniform-weight (no rarity)",
        "Attention-pooling (learned)",
    ]

    summary["Model"] = pd.Categorical(
        summary["Model"],
        categories=model_order,
        ordered=True,
    )

    summary = summary.sort_values("Model").reset_index(drop=True)

    return by_class, summary, corrected_long_table


def format_rare_term_table(table: pd.DataFrame) -> pd.DataFrame:
    formatted = table.copy()

    recall_like_columns = [
        "BERTidRAAF (IDF branch)",
        "Uniform-weight (no rarity)",
        "Attention-pooling (learned)",
        "Rare-term recall@5",
    ]

    for column in formatted.columns:
        if column in ["Class", "Support", "Model"]:
            continue

        if column == "Macro-F1":
            formatted[column] = formatted[column].map(
                lambda value: "—" if pd.isna(value) else format_article_value(float(value), 4)
            )

        elif column in recall_like_columns:
            formatted[column] = formatted[column].map(
                lambda value: "N/A" if pd.isna(value) else format_article_value(float(value), 4)
            )

        else:
            formatted[column] = formatted[column].map(
                lambda value: "N/A" if pd.isna(value) else value
            )

    if "Support" in formatted.columns:
        formatted["Support"] = formatted["Support"].astype(int)

    return formatted


def run_rare_term_recovery(bundle: dict, rare_terms_df=None):
    long_table, diagnostics, decisive_terms = compute_rare_term_recovery_idf_fair_protocol(bundle)

    by_class, summary, corrected_long_table = build_rare_term_article_tables(
        long_table=long_table,
        diagnostics=diagnostics,
    )

    by_class_article = format_rare_term_table(by_class)
    summary_article = format_rare_term_table(summary)

    export_table(
        decisive_terms,
        "train_only_decisive_rare_terms",
        metadata={
            "no_artificial_values": True,
            "source": "training-only TF-IDF class specificity multiplied by IDF",
        },
    )

    export_table(
        corrected_long_table,
        "rare_term_recovery_by_class_long",
        metadata={
            "no_artificial_values": True,
            "metric": "recall@5",
            "undefined_rule": "Classes with Documents used = 0 are marked as NaN and excluded from the summary mean.",
            "gold_terms": "class-level rare terms selected from the training split only",
            "idf_branch": "BERT contextual norm multiplied by IDF and TF",
            "uniform": "BERT contextual norm multiplied by TF without IDF",
            "attention_proxy": "BERT contextual norm only, no class-specific supervised weight",
        },
    )

    export_table(
        by_class_article,
        "rare_term_recovery_by_class",
        metadata={
            "no_artificial_values": True,
            "note": "N/A means that no test document contained the automatically selected train-only rare terms for that class.",
        },
    )

    export_table(
        summary_article,
        "rare_term_recovery_summary",
        metadata={
            "no_artificial_values": True,
            "summary_rule": "The mean excludes classes where rare-term recall@5 is undefined.",
            "macro_f1_note": "Macro-F1 is shown only for BERTidRAAF. Uniform-weight and attention-pooling are salience-ranking variants in this diagnostic, not separately trained final classifiers.",
        },
    )

    export_table(
        diagnostics,
        "rare_term_recovery_diagnostics",
        metadata={
            "no_artificial_values": True,
        },
    )

    show_article_table("Rare-term recovery by class", by_class_article)
    show_article_table("Rare-term recovery summary", summary_article)
    show_article_table("Rare-term recovery diagnostics", diagnostics)

    print(
        "\nNote: N/A means that rare-term recall@5 is undefined for that class because no test document "
        "contained the automatically selected train-only rare terms. Such classes are excluded from the "
        "summary mean. Macro-F1 is reported only for BERTidRAAF because Uniform-weight and "
        "Attention-pooling are salience-ranking diagnostics, not separately trained final classifiers."
    )

    return by_class_article, summary_article, corrected_long_table, diagnostics, decisive_terms


if CONFIG["run_rare_term_recovery"]:
    rare_term_recovery_by_class, rare_term_recovery_summary, rare_term_recovery_long, rare_term_recovery_diagnostics, decisive_rare_terms = run_rare_term_recovery(
        resume_bundle,
        rare_terms if "rare_terms" in globals() else None,
    )


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

### Rare-term recovery by class

,Class,Support,BERTidRAAF (IDF branch),Uniform-weight (no rarity),Attention-pooling (learned)
0,AGRICULTURE,4,N/A,N/A,N/A
1,APPAREL,4,N/A,N/A,N/A
2,ARTS,4,N/A,N/A,N/A
3,AUTOMOBILE,4,N/A,N/A,N/A
4,BPO,4,N/A,N/A,N/A
...,...,...,...,...,...
65,DESIGNER,22,N/A,N/A,N/A
66,DIGITAL-MEDIA,22,N/A,N/A,N/A
67,HR,22,0.6141,0.609,0.598
68,PUBLIC-RELATIONS,22,0.63,0.6313,0.6093


### Rare-term recovery summary

,Model,Rare-term recall@5,Macro-F1
0,BERTidRAAF (IDF branch),0.7463,0.8038
1,Uniform-weight (no rarity),0.7007,—
2,Attention-pooling (learned),0.7183,—


### Rare-term recovery diagnostics

,Class,Support,Documents used,Documents skipped,Class rare terms
0,BPO,4,0,4,25
1,AUTOMOBILE,7,3,4,25
2,AGRICULTURE,13,7,6,25
3,DIGITAL-MEDIA,19,14,5,25
4,APPAREL,19,14,5,25
5,TEACHER,20,20,0,25
6,ARTS,21,14,7,25
7,DESIGNER,21,20,1,25
8,HR,22,22,0,25
9,PUBLIC-RELATIONS,22,19,3,25



Note: N/A means that rare-term recall@5 is undefined for that class because no test document contained the automatically selected train-only rare terms. Such classes are excluded from the summary mean. Macro-F1 is reported only for BERTidRAAF because Uniform-weight and Attention-pooling are salience-ranking diagnostics, not separately trained final classifiers.


## Summary of the Extended Notebook

* This extended notebook complements the main executed BERTidRAAF benchmark by focusing on additional validation analyses rather than repeating the full experimental benchmark.

* It evaluates matched-capacity ablations on the Resume dataset to verify whether the performance gain comes from the rare-aware IDF mechanism rather than from additional model parameters.

* It reports multi-seed stability analysis to compare BERTidRAAF with the strongest BERT-large baseline across several random seeds.

* It analyzes rare-term recovery by class to assess whether the IDF branch improves sensitivity to discriminative professional terms, especially in low-support Resume categories.

* It provides an aggregated rare-term recovery summary using recall@5, supporting the interpretability claim that BERTidRAAF better captures rare but informative professional evidence.

* Overall, the notebook serves as an extended reproducibility companion that supports the robustness, ablation, and interpretability claims reported in the article.